# Landscript: Glyph Extraction Pipeline

Discovers letter-like shapes in satellite imagery.

**Quick start (local):** `python run_pipeline.py --region bangalore --scenes 3`

This notebook walks through the same steps manually.

In [ ]:
# @title 1. Install dependencies (Colab only — skip if running locally)
# !pip install -e /content/landscript
print('Run this only in Colab after cloning the repo')

In [ ]:
# @title 2. Configure pipeline
from landscript.config import REGIONS, PipelineConfig

cfg = PipelineConfig(
    bbox=REGIONS["bangalore"],
    region_name="bangalore",
    state="Karnataka",
    satellite="sentinel-2",
    cloud_cover_max=20,
    similarity_threshold=0.15,
)
print(f'Region: {cfg.region_name}')
print(f'Satellite: {cfg.satellite}')
print(f'Bounds: {cfg.bbox}')

In [ ]:
# @title 3. Search available scenes
from landscript.stac import list_scenes
scenes = list_scenes(cfg)
print(f'Found {len(scenes)} scenes')
for s in scenes[:10]:
    print(f"  {s['date']} | cloud: {s['cloud']:.0f}% | {s['satellite']}")

In [ ]:
# @title 4. Download and tile best scenes
from landscript.stac import download_and_tile
tile_files = download_and_tile(cfg, max_scenes=3)
print(f'{len(tile_files)} tiles ready')

In [ ]:
# @title 5. Load letter templates and process tiles
from landscript.cv_pipeline import (
    load_letter_templates, to_grayscale, apply_threshold,
    find_contours, filter_contours, match_shape,
    contour_to_polygon, extract_glyph_crop,
)
from landscript.metadata import GlyphStore
import cv2

templates = load_letter_templates(cfg)
store = GlyphStore(cfg.metadata_dir / f'{cfg.region_name}.json')
print(f'{len(templates)} templates, {store.count()} existing glyphs')

found = 0
for tile_path in tile_files:
    img = cv2.imread(str(tile_path))
    if img is None:
        continue
    gray = to_grayscale(img)
    binary = apply_threshold(gray, cfg.threshold_method)
    contours = find_contours(binary)
    contours = filter_contours(contours, cfg)
    for contour in contours:
        poly = contour_to_polygon(contour, cfg.epsilon)
        for tmpl in templates:
            score = match_shape(poly, tmpl.contour)
            if score >= cfg.similarity_threshold:
                continue
            x, y, w, h = cv2.boundingRect(contour)
            glyph_id = store.add({
                'letter': tmpl.letter,
                'score': float(score),
                'bbox': {'x': x, 'y': y, 'w': w, 'h': h},
                'source_tile': tile_path.name,
                'region': cfg.region_name,
                'state': cfg.state,
                'country': cfg.country,
            })
            extract_glyph_crop(img, contour, cfg.glyphs_dir / tmpl.letter / f'{glyph_id}.png')
            found += 1

print(f'Found {found} glyph candidates')

In [ ]:
# @title 6. Browse results
letter = 'A'
results = store.search(letter=letter, limit=10)
print(f"{len(results)} candidates for '{letter}'")
for r in results:
    path = cfg.glyphs_dir / r['letter'] / f"{r['id']}.png"
    if path.exists():
        from IPython.display import display, Image
        print(f"{r['id']}: score={r['score']:.4f}")
        display(Image(filename=str(path)))

In [ ]:
# @title Stats
all_g = store.all()
accepted = [g for g in all_g if g.get('accepted')]
letters = set(g.get('letter') for g in all_g if g.get('letter'))
print(f'Total: {len(all_g)} | Accepted: {len(accepted)} | Letters: {sorted(letters)}')